# M* BAGEL: pinned single-image baseline

This notebook verifies one unchanged, single-GPU BAGEL text-to-image request on an **A100 80 GB High-RAM** Colab runtime. It is a setup/correctness milestone, not a performance experiment. The server is stopped in a `finally` block and results are saved to Google Drive.

In [ ]:
from google.colab import drive
from datetime import datetime, timezone
from pathlib import Path
import json, os, subprocess, sys

drive.mount('/content/drive')
RUN_ID = 'smoke_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
FORK_URL = 'https://github.com/yegurla-sumanth/mstar.git'
GIT_REF = 'bagel-baseline'
MSTAR_BASE = '7fa8b8e72e26097979805eca8fe40930e088ff64'
REPO = Path('/content') / f'mstar_{RUN_ID}'
CACHE_DIR = Path('/content/hf_cache_bagel')
RESULT_DIR = Path('/content/drive/MyDrive/CMU_11868_BAGEL') / RUN_ID
RESULT_DIR.mkdir(parents=True, exist_ok=False)
print({'run_id': RUN_ID, 'repo': str(REPO), 'results': str(RESULT_DIR)})

In [ ]:
# Clone first, then reject the wrong GPU/storage before installing or downloading weights.
subprocess.run(['git', 'clone', '--branch', GIT_REF, '--single-branch', FORK_URL, str(REPO)], check=True)
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
subprocess.run(['git', 'merge-base', '--is-ancestor', MSTAR_BASE, head], cwd=REPO, check=True)
source = {
    'fork_url': FORK_URL,
    'requested_ref': GIT_REF,
    'resolved_commit': head,
    'base_commit': MSTAR_BASE,
    'git_status': subprocess.check_output(['git', 'status', '--porcelain=v1'], cwd=REPO, text=True),
}
(RESULT_DIR / 'source.json').write_text(json.dumps(source, indent=2) + '\n')
probe = REPO / 'experiments/bagel/scripts/environment_probe.py'
subprocess.run([
    sys.executable, str(probe),
    '--output', str(RESULT_DIR / 'preinstall_environment.json'),
    '--disk-path', '/content',
    '--require-gpu-memory-gib', '75',
    '--require-host-memory-gib', '50',
    '--require-disk-free-gib', '70',
], check=True)
print('Preflight passed at commit', head)

In [ ]:
# Create an isolated Python 3.12 environment and install the candidate CUDA 12.8 stack.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'uv==0.10.0'], check=True)
UV = subprocess.check_output([sys.executable, '-c', 'import shutil; print(shutil.which(\"uv\"))'], text=True).strip()
VENV = REPO / '.venv-bagel'
subprocess.run([UV, 'venv', '--python', '3.12', '--seed', str(VENV)], check=True)
VENV_PY = VENV / 'bin/python'
constraints = REPO / 'experiments/bagel/constraints-colab-cu128.txt'
subprocess.run([
    UV, 'pip', 'install', '--python', str(VENV_PY), '--torch-backend=cu128',
    '--constraint', str(constraints), '-e', f'{REPO}[bagel]',
], check=True)
subprocess.run([UV, 'pip', 'check', '--python', str(VENV_PY)], check=True)
ninja_path = VENV / 'bin/ninja'
assert ninja_path.exists(), f'Missing required FlashInfer JIT tool: {ninja_path}'
subprocess.run([str(ninja_path), '--version'], check=True)
subprocess.run([str(VENV_PY), str(REPO / 'test/modular/test_bagel_checkpoint_revision.py')], cwd=REPO, check=True)
import_check = subprocess.check_output([
    str(VENV_PY), '-c',
    'import torch, flashinfer, transformers; assert torch.cuda.is_available(); x=torch.ones((16,16),device=\"cuda\",dtype=torch.bfloat16); y=x@x; torch.cuda.synchronize(); print(torch.__version__, torch.version.cuda, flashinfer.__version__, transformers.__version__, float(y[0,0]))'
], text=True)
(RESULT_DIR / 'gpu_import_check.txt').write_text(import_check)
freeze = subprocess.check_output([UV, 'pip', 'freeze', '--python', str(VENV_PY)], text=True)
(RESULT_DIR / 'resolved_requirements.txt').write_text(freeze)
subprocess.run([
    str(VENV_PY), str(REPO / 'experiments/bagel/scripts/environment_probe.py'),
    '--output', str(RESULT_DIR / 'installed_environment.json'), '--disk-path', '/content',
    '--require-gpu-memory-gib', '75', '--require-host-memory-gib', '50',
    '--require-disk-free-gib', '60',
], check=True)
subprocess.run([str(VENV / 'bin/mstar-serve'), '--help'], check=True, stdout=subprocess.DEVNULL)
print('Environment installation and import preflight passed')

In [ ]:
# Download exactly the checkpoint revision named in the pinned server config.
subprocess.run([
    str(VENV_PY), str(REPO / 'experiments/bagel/scripts/download_checkpoint.py'),
    '--cache-dir', str(CACHE_DIR),
    '--manifest', str(RESULT_DIR / 'checkpoint_manifest.json'),
], check=True)
import yaml
manifest = json.loads((RESULT_DIR / 'checkpoint_manifest.json').read_text())
pinned_config = yaml.safe_load((REPO / 'experiments/bagel/configs/bagel_single_gpu_pinned.yaml').read_text())
assert manifest['resolved_revision'] == pinned_config['model_kwargs']['checkpoint_revision']
print('Pinned checkpoint is available at', manifest['resolved_revision'])

In [ ]:
# The orchestrator starts M*, waits for health, runs one request, and stops all server children.
SMOKE_DIR = RESULT_DIR / 'single_image'
subprocess.run([
    str(VENV_PY), str(REPO / 'experiments/bagel/scripts/run_smoke.py'),
    '--repo-root', str(REPO),
    '--cache-dir', str(CACHE_DIR),
    '--output-dir', str(SMOKE_DIR),
], check=True)
print('Smoke orchestrator completed')

In [ ]:
from IPython.display import Image, display
smoke = json.loads((SMOKE_DIR / 'smoke_result.json').read_text())
run = json.loads((SMOKE_DIR / 'run_summary.json').read_text())
assert smoke['status'] == 'success', smoke
assert run['status'] == 'success', run
display(Image(filename=str(SMOKE_DIR / 'image.png')))
print(json.dumps({
    'resolved_commit': head,
    'client_elapsed_seconds': smoke['client_elapsed_seconds'],
    'server_ready_seconds': run['server_ready_seconds'],
    'peak_sampled_gpu_memory_mib': smoke['gpu_sampling']['peak_memory_used_mib'],
    'image_sha256': smoke['output']['sha256'],
    'shutdown': run['shutdown'],
    'result_dir': str(RESULT_DIR),
}, indent=2))

After reviewing the displayed image and saved summary, use **Runtime → Disconnect and delete runtime**. The M* server has already been terminated; disconnecting releases the Colab GPU.